### Notebook Description
> version: `ipynb-feat-v01.00.00`

This notebook is designed to **set up and run an automated futures trading system** on the AIFAPBT platform. It is configured for the Google Colab environment.

**Key Steps:**

*   **Mount Google Drive:** Connect to Google Drive to access strategy and configuration files.
*   **Register API Key:** Set up your User API key to access the trading platform.
*   **Upload Files:** Upload your module, strategy `.py` file, and YAML configuration to the server.
*   **Validate Strategy:** Verify the uploaded strategy and check weight constraints before trading.
*   **Run Auto-Trade:** Execute the automated trading system. Outputs the session ID and dashboard URL.
*   **Terminate:** Forcefully terminate the running automated trading system.
*   **Final Check:** Check remaining assets and perform forced liquidation for positions greater than 1 USDT.

**Note:** You must execute the terminate step before performing forced liquidation of the current process.

#### Original GitHub Link
- https://github.com/NeoMatrixAI/nb-runner
- https://github.com/NeoMatrixAI/strategy

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore')

# Register Api Key

In [ ]:
USER_KEY = " "      #  Input your User key

print("USER_KEY:", USER_KEY if USER_KEY else "Not found")

# Health check

In [ ]:
import requests

root_url = f'https://aifapbt.fin.cloud.ainode.ai/'
headers = {"API-KEY": USER_KEY}
requests.get(root_url, headers=headers).json()

In [ ]:
tradeType = "futures"
strategy_name = "multi_period_momentum"

# 1. Upload Module

In [ ]:
import os

# Upload or Update module file
endpoint = 'upload/module'
url = root_url + endpoint

module_name = "momentum"

base_path = "/content/drive/MyDrive/NeoMatrixAI/common"
file_path = os.path.join(base_path, module_name + ".py")

params = {'overwrite': True} # If overwrite is True, it will modify the existing file.

with open(file_path, "rb") as f:
    files = {"file": f}
    response = requests.post(url, headers=headers, params=params, files=files)

print("📂 Upload Response:", response.json())

# 2. Upload Strategy

In [ ]:
# Upload or Update strategy file
endpoint = 'upload/strategy'
url = root_url + endpoint

base_path = f"/content/drive/MyDrive/NeoMatrixAI/{tradeType}"
file_path = os.path.join(base_path, strategy_name, strategy_name + ".py")

params = {'tradeType': tradeType, 'overwrite': True} # If overwrite is True, it will modify the existing file.

with open(file_path, "rb") as f:
    files = {"file": f}
    response = requests.post(url, headers=headers, params=params, files=files)

print("📂 Upload Response:", response.json())

# 3. Upload configs

In [ ]:
# Upload YAML Config
endpoint = 'upload/config'
url = root_url + endpoint

# Workspace path
base_path = f"/content/drive/MyDrive/NeoMatrixAI/{tradeType}"
file_path = os.path.join(base_path, strategy_name, "config.yaml")

params = {'tradeType': tradeType, 'overwrite': True}

with open(file_path, "rb") as f:
    files = {"file": f}
    response = requests.post(url, headers=headers, params=params, files=files)

print("📂 Config Upload Response:", response.json())

# 4. Validate strategy

In [ ]:
endpoint = 'upload/validate-strategy'
url = root_url + endpoint

params = {'tradeType': tradeType, 'strategy_name': strategy_name}

response = requests.post(url, headers=headers, params=params)
print(response)
result = response.json()

if result['valid']:
    print("✅ Validation Success!")
    print(f"📊 Weights: {result['weights']}")
    if result['warnings']:
        print(f"⚠️ Warning: {result['warnings']}")
else:
    print("❌ Validation Failed!")
    for err in result['errors']:
        print(f"  [{err['step']}] {err['message']}")
    print(f"  Completed steps: {result['steps_completed']}")

    
result['weights']

# 4. Run Auto-Trade

In [ ]:
# run-system
endpoint = 'command/run-system'
url = root_url + endpoint

data = {
    "tradeType": tradeType,
    "strategy_name": strategy_name,
    "trade_mode": "rebalancing"
}
    
response = requests.post(url, headers=headers, json=data)
print(response.json())
try:
    session_id = response.json()['session_id']
    print('session_id :', response.json()['session_id'])
    print('dashboard :', response.json()['dashboard_url'])
except:
    print('Error :', response.json()['message']['message'])
    session_id = response.json()['message']['session_id']
    print('session_id id :', session_id)


# Terminate

In [ ]:
# Terminate process
# You must execute terminate to force liquidation of the current process.
endpoint = 'command/terminate'
url = root_url + endpoint

params = {"session_id": f"{session_id}", "tradeType": tradeType}

response = requests.get(url, headers=headers, params=params)
print(response.json())

# Final Check

### Check remaining assets after terminating

In [ ]:
from drive.MyDrive.NeoMatrixAI.module.futures import position, trade

df_all_position = position.all_positions(USER_KEY, 'live', 'usdt-futures', 'USDT')
df_all_position

### ⚠️ Forced Liquidation
- If the position value for each asset is greater than 1, forced liquidation is performed.

In [ ]:
print(f"Found {len(df_all_position)} assets.")

try:
    df_res = trade.flash_close_position(USER_KEY, 'live', None, 'usdt-futures', None)
    display(df_res)
    
except:
    print(f"No assets found in account. Nothing to sell.")